# Conversational Clustering — Week 4: Closing the loop (demo)

**Goal:** demonstrate the full conversational clustering loop end-to-end on a single run, for the upcoming presentation. Not the full experiment — just the proof that the mechanism works.

**Scope decisions for the demo:**

| Choice | Value | Why |
|---|---|---|
| Target | `topic` (arXiv primary categories) | Free labels, easiest to demonstrate measurable progress |
| Starting clustering | Week 2 `generic` baseline (ARI=0.245 vs arXiv) | Real starting point, room to improve |
| Operations | `split`, `merge`, `change_k` | The three core ops; `re-embed` and `ignore` left as stretch |
| Multiple ops per turn | yes | More realistic — one feedback message often implies several changes |
| Turn budget | 6 | Same as Week 3 design |
| Number of runs | 1 | This is a demo, not an experiment |

**What this notebook will NOT do:** stats, multiple runs, confidence intervals, three-way system comparison. All of that is the *real* experiment, which is post-presentation work.

**Expected demo outcome:**
- Starting ARI ≈ 0.245
- ARI rises (with possible oscillations) as turns progress
- After 6 turns, ARI somewhere in [0.30, 0.50] would be a great demo result
- Even if it doesn't beat the 0.361 topic baseline from Week 2, that's fine — the point is to show the loop closes

---

## Router v0.2 — what changed from v0.1

The first demo run (router v0.1) ended with ARI = 0.199 (down from 0.245 initial). Diagnosis: the router responded to every user complaint by increasing K and splitting clusters, with no awareness that it was already over-fragmenting. K went 5 → 6 → 9 → 13 → 18 → 22 → 18. Since the topic axis has ~6 categories, K > 8 inevitably hurts ARI vs arXiv.

**Three targeted changes in v0.2:**

1. **K discipline in the router prompt.** Explicit guidance: K should stay in [5, 8] for topic; do not increase K above 8 without strong justification; repeated change_k upward is usually a mistake.
2. **Router sees its own history.** Past K values and past operations are shown to the router each turn, so it can detect "I've already increased K twice and the clustering hasn't improved" patterns.
3. **2-op cap per turn.** Was 4 in v0.1. Fewer ops = more stable, fewer cascading effects. Enforced both in the prompt and server-side as defense-in-depth.

The simulated user is unchanged. We only modify one side of the loop at a time so we can attribute the improvement.


## 0. Setup

In [ ]:
import os
import json
import time
import hashlib
import re
import copy
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
CACHE_DIR = Path("cache_claude")
CACHE_DIR.mkdir(exist_ok=True)
ABSTRACTS_PATH = DATA_DIR / "astro_ph_abstracts.json"

ANTHROPIC_MODEL = "claude-sonnet-4-6"

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"


In [ ]:
# call_llm copied from Week 2/3 (same logic, same cache).
def _cache_key(static_block, varying, model):
    h = hashlib.sha256(f"{model}|||{static_block}|||{varying}".encode()).hexdigest()[:16]
    return f"{model}_{h}"

def call_llm(static_block, varying, max_tokens=2048, temperature=0.0, use_cache=True):
    model = ANTHROPIC_MODEL
    cache_file = CACHE_DIR / f"{_cache_key(static_block, varying, model)}.json"
    if use_cache and cache_file.exists():
        with open(cache_file) as f:
            payload = json.load(f)
        payload["cached"] = True
        return payload

    import anthropic
    client = anthropic.Anthropic()
    t0 = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        system=[{"type": "text", "text": static_block, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": varying}],
    )
    duration = time.time() - t0
    text = response.content[0].text
    usage = {
        "input_tokens": response.usage.input_tokens,
        "cache_creation_input_tokens": getattr(response.usage, "cache_creation_input_tokens", 0) or 0,
        "cache_read_input_tokens": getattr(response.usage, "cache_read_input_tokens", 0) or 0,
        "output_tokens": response.usage.output_tokens,
    }
    result = {"response": text, "usage": usage, "model": model, "cached": False, "duration_s": duration}
    if use_cache:
        with open(cache_file, "w") as f:
            json.dump(result, f)
    return result


def estimate_cost(usage):
    r_in = 3.0 / 1_000_000
    r_out = 15.0 / 1_000_000
    return (usage["input_tokens"] * r_in
            + usage["cache_creation_input_tokens"] * r_in * 1.25
            + usage["cache_read_input_tokens"] * r_in * 0.10
            + usage["output_tokens"] * r_out)


## 1. Load corpus, embeddings, and the starting clustering

Embeddings come from Week 1 (sentence-transformer all-MiniLM-L6-v2). We need them because two operations (`split`, plus the cluster summarizer) work in embedding space.


In [ ]:
from sentence_transformers import SentenceTransformer

# Load abstracts
with open(ABSTRACTS_PATH) as f:
    abstracts = json.load(f)
df = pd.DataFrame(abstracts)
print(f"Loaded {len(df)} abstracts")

# Embed (cached internally by sentence-transformers between sessions if reload)
print("Computing embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [f"{r.title}. {r.abstract}" for r in df.itertuples(index=False)]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print(f"Embeddings: {embeddings.shape}")

# Load Week 2 generic baseline as the starting clustering
with open(DATA_DIR / "claude_baseline_generic.json") as f:
    baseline_generic = json.load(f)
initial_assignments = np.array(baseline_generic["assignments"])
initial_labels = {int(k): v for k, v in baseline_generic["labels"].items()}

# Target = arXiv primary categories (topic axis)
primary_cats = df["primary_category"].tolist()
cat_to_int = {c: i for i, c in enumerate(sorted(set(primary_cats)))}
target_assignments = np.array([cat_to_int[c] for c in primary_cats])

initial_ari = adjusted_rand_score(target_assignments, initial_assignments)
print(f"\nStarting clustering: {len(set(initial_assignments))} clusters")
print(f"Labels: {list(initial_labels.values())}")
print(f"Initial ARI vs arXiv topic target: {initial_ari:.3f}")


## 2. State container

A simple dataclass to track the clustering state across turns. We pass this around instead of juggling individual numpy arrays.


In [ ]:
@dataclass
class ClusteringState:
    assignments: np.ndarray  # shape (n_papers,)
    labels: dict             # {cluster_id: label_string}
    history: list = field(default_factory=list)  # log of operations applied

    def k(self):
        return len(set(int(c) for c in self.assignments if c >= 0))

    def copy(self):
        return ClusteringState(
            assignments=self.assignments.copy(),
            labels=dict(self.labels),
            history=list(self.history),
        )

    def renumber(self):
        """Compact cluster IDs to 0..K-1 after split/merge operations may have left gaps."""
        old_ids = sorted(set(int(c) for c in self.assignments if c >= 0))
        remap = {old: new for new, old in enumerate(old_ids)}
        new_assignments = np.array([remap.get(int(c), -1) for c in self.assignments])
        new_labels = {remap[old]: self.labels[old] for old in old_ids if old in self.labels}
        self.assignments = new_assignments
        self.labels = new_labels


## 3. The three operations

Cluster-level ops. None of them take individual paper IDs — that's intentional, to avoid the cheating risk the simulated-user prompt is designed to avoid.

- **`change_k(state, new_k)`** — re-run k-means with new K on all papers. Labels are re-generated by the LLM cluster-summarizer.
- **`merge(state, c_a, c_b)`** — combine two clusters. New label = LLM-generated from the merged members.
- **`split(state, c, criterion)`** — sub-cluster the members of one cluster into 2 using k-means on their embeddings. The `criterion` is a *string description* of how to split, used to re-label the result (but the actual partitioning is k-means in embedding space — it's a known limitation for the demo).


In [ ]:
def _llm_label_cluster(member_indices: np.ndarray, df: pd.DataFrame, hint: str = None, use_cache: bool = True) -> str:
    """Ask the LLM for a short label for a set of papers."""
    titles = [df.iloc[i]["title"] for i in member_indices[:15]]  # cap at 15 titles
    titles_block = "\n".join(f"- {t}" for t in titles)
    static = "You are labeling a cluster of academic papers. You will be given a list of paper titles and asked for a concise label (2-5 words)."
    varying = (
        (f"Hint: this group is about {hint}. " if hint else "")
        + f"\n\nTitles in this cluster:\n{titles_block}\n\n"
        + "Respond with ONLY a 2-5 word label, no quotes, no preamble."
    )
    out = call_llm(static, varying, max_tokens=30, use_cache=use_cache)
    label = out["response"].strip().strip('"').strip("'")
    # Sanity: cap length
    if len(label) > 60:
        label = label[:60]
    return label


def op_change_k(state: ClusteringState, new_k: int, use_cache: bool = True) -> ClusteringState:
    """Re-run k-means with new_k clusters."""
    new_state = state.copy()
    km = KMeans(n_clusters=new_k, random_state=RANDOM_SEED, n_init=5)
    new_state.assignments = km.fit_predict(embeddings)
    # Relabel each new cluster
    new_labels = {}
    for cid in range(new_k):
        members = np.where(new_state.assignments == cid)[0]
        if len(members) == 0:
            new_labels[cid] = f"cluster_{cid}"
            continue
        new_labels[cid] = _llm_label_cluster(members, df, use_cache=use_cache)
    new_state.labels = new_labels
    new_state.history.append({"op": "change_k", "new_k": new_k})
    return new_state


def op_merge(state: ClusteringState, c_a: int, c_b: int, use_cache: bool = True) -> ClusteringState:
    """Merge cluster c_b into c_a, remove c_b."""
    new_state = state.copy()
    if c_a not in new_state.labels or c_b not in new_state.labels:
        new_state.history.append({"op": "merge", "c_a": c_a, "c_b": c_b, "skipped": "cluster doesn't exist"})
        return new_state
    if c_a == c_b:
        new_state.history.append({"op": "merge", "c_a": c_a, "c_b": c_b, "skipped": "same cluster"})
        return new_state
    new_state.assignments = np.where(new_state.assignments == c_b, c_a, new_state.assignments)
    new_state.labels.pop(c_b, None)
    # Re-label merged cluster
    members = np.where(new_state.assignments == c_a)[0]
    new_state.labels[c_a] = _llm_label_cluster(members, df, use_cache=use_cache)
    new_state.renumber()
    new_state.history.append({"op": "merge", "c_a": c_a, "c_b": c_b})
    return new_state


def op_split(state: ClusteringState, c: int, criterion: str = None, use_cache: bool = True) -> ClusteringState:
    """Sub-cluster the members of cluster c into 2 sub-clusters via k-means on embeddings.

    Demo limitation: the actual split is k-means in embedding space; the `criterion`
    string is only used as a labeling hint, not to actually partition. Acknowledged
    weakness — see notebook discussion.
    """
    new_state = state.copy()
    if c not in new_state.labels:
        new_state.history.append({"op": "split", "c": c, "skipped": "cluster doesn't exist"})
        return new_state
    members = np.where(new_state.assignments == c)[0]
    if len(members) < 4:
        new_state.history.append({"op": "split", "c": c, "skipped": f"too small ({len(members)} members)"})
        return new_state

    sub_embs = embeddings[members]
    km = KMeans(n_clusters=2, random_state=RANDOM_SEED, n_init=5)
    sub_assignments = km.fit_predict(sub_embs)

    # Allocate new cluster ID
    new_cid = max(new_state.labels.keys()) + 1
    # First sub-cluster keeps the original ID, second gets the new one
    for j, mem_idx in enumerate(members):
        if sub_assignments[j] == 1:
            new_state.assignments[mem_idx] = new_cid

    # Re-label both halves
    members_a = np.where(new_state.assignments == c)[0]
    members_b = np.where(new_state.assignments == new_cid)[0]
    new_state.labels[c] = _llm_label_cluster(members_a, df, hint=criterion, use_cache=use_cache)
    new_state.labels[new_cid] = _llm_label_cluster(members_b, df, hint=criterion, use_cache=use_cache)
    new_state.history.append({"op": "split", "c": c, "criterion": criterion, "new_cid": new_cid})
    return new_state


## 4. The LLM router

Reads the simulated user's feedback + the current clustering state, emits a list of operations as structured JSON.

The router is given the **operation vocabulary** as part of its system prompt. It cannot invent operations outside this set. Output is JSON for reliable parsing.


In [ ]:
# Router v0.2 — adds K guardrails and visibility into past turns.
# Changes from v0.1:
#   - Hard guidance against over-fragmentation (K > 8 only with strong justification)
#   - Router now sees past K and past operations, so it can detect anti-patterns
#   - Operations capped at 2 per turn (was 4) — more stable, fewer cascading effects
ROUTER_SYSTEM_PROMPT = '''You are the router for a conversational clustering system. The user (an astronomer) just gave you natural-language feedback about the current clustering. Your job is to translate that feedback into a SHORT list of typed operations on the clustering pipeline.

The operations you can emit:

1. `change_k` with `new_k`: re-run k-means with a different number of clusters.
   Example: {{"op": "change_k", "new_k": 6}}

2. `merge` with `c_a` and `c_b`: combine cluster c_b into c_a.
   Example: {{"op": "merge", "c_a": 1, "c_b": 4}}

3. `split` with `c` and `criterion`: split cluster `c` into two sub-clusters.
   `criterion` is a short description of how to split (used as a labeling hint).
   Example: {{"op": "split", "c": 2, "criterion": "separate observational from theoretical"}}

Current clustering:

{clustering_view}

{history_section}

GUIDELINES:

- Emit AT MOST 2 operations per turn. Fewer is better. One well-chosen op beats four reactive ones.
- Match what the user is asking for, not what you think is best.

K DISCIPLINE (this is important — past runs over-fragmented and broke convergence):
- The target axis (topic) has ~6 natural categories. K should stay roughly in [5, 8].
- DO NOT increase K above 8 unless the user is explicitly asking for very fine-grained clusters.
- If the current K is already >= 8 and the user reports problems, prefer `merge` and `split` of specific clusters over another `change_k` increase.
- If past turns have increased K and the clustering has not improved, that is a signal to MERGE, not split further.
- Repeated `change_k` upward is usually a mistake — each one re-runs k-means from scratch and may erase progress.

CONSERVATIVE BEHAVIOR:
- Prefer surgical operations (split or merge of one specific cluster) over global ones (change_k).
- If the user reports mixing within ONE cluster, use `split` on that cluster.
- If the user wants TWO clusters combined, use `merge`.
- Only use `change_k` if K is fundamentally wrong (e.g., K=3 when 6 categories are needed).

- Use existing cluster IDs from the current clustering. Do not invent new ones.
- If feedback is genuinely too vague, emit zero operations: {{"operations": []}}.

OUTPUT FORMAT — JSON only, no prose:

{{
  "reasoning": "<short explanation, 1-2 sentences, mentioning if you considered past turns>",
  "operations": [
    {{"op": "...", ...}}
  ]
}}'''


ROUTER_USER_PROMPT = '''User feedback (turn {turn_idx}):

"{feedback}"

Emit JSON with reasoning and operations (at most 2).'''


def render_router_history(past_turns: list) -> str:
    """Show the router its past decisions, so it can detect anti-patterns."""
    if not past_turns:
        return ""
    lines = ["Recent history of this conversation (your past decisions):"]
    for t in past_turns:
        k_before = t.get("k_before", "?")
        k_after = t.get("k_after", "?")
        ops_summary = ", ".join(op.get("op", "?") for op in t.get("operations", []))
        if not ops_summary:
            ops_summary = "no operations"
        lines.append(f"  - Turn {t['turn']}: K {k_before} → {k_after}, ops: [{ops_summary}]")
    lines.append("")
    lines.append("Use this history to avoid repeating ineffective patterns (especially: repeated K increases without improvement).")
    return "\n".join(lines)


def render_clustering_view_with_ids(state: ClusteringState, df: pd.DataFrame, samples_per_cluster: int = 4) -> str:
    """Like the user-facing view, but the router gets to see cluster IDs (it needs them)."""
    rng = np.random.RandomState(RANDOM_SEED)
    lines = []
    for cid in sorted(state.labels.keys()):
        members = np.where(state.assignments == cid)[0]
        if len(members) == 0:
            continue
        label = state.labels[cid]
        lines.append(f"Cluster {cid}: \"{label}\" ({len(members)} papers)")
        sample_idx = rng.choice(members, size=min(samples_per_cluster, len(members)), replace=False)
        for idx in sorted(sample_idx):
            t = df.iloc[idx]["title"]
            if len(t) > 95:
                t = t[:92] + "..."
            lines.append(f"  - {t}")
        lines.append("")
    return "\n".join(lines).strip()


def router_decide_operations(
    state: ClusteringState,
    feedback: str,
    turn_idx: int,
    router_history: list = None,
    use_cache: bool = True,
) -> dict:
    """Ask the router to translate feedback into operations."""
    view = render_clustering_view_with_ids(state, df)
    history_section = render_router_history(router_history or [])
    static = ROUTER_SYSTEM_PROMPT.format(
        clustering_view=view,
        history_section=history_section,
    )
    varying = ROUTER_USER_PROMPT.format(turn_idx=turn_idx, feedback=feedback)

    llm_out = call_llm(static, varying, max_tokens=512, temperature=0.0, use_cache=use_cache)
    text = llm_out["response"].strip()

    # Strip code fences if present
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.MULTILINE).strip()

    try:
        parsed = json.loads(text)
        operations = parsed.get("operations", [])
        reasoning = parsed.get("reasoning", "")
    except json.JSONDecodeError as e:
        operations = []
        reasoning = f"PARSE FAILURE: {e}"

    # Defense in depth: even if the prompt-level cap is ignored, enforce max 2 ops per turn.
    truncated = False
    if len(operations) > 2:
        operations = operations[:2]
        truncated = True

    return {
        "operations": operations,
        "reasoning": reasoning + (" [truncated to 2 ops]" if truncated else ""),
        "raw_response": text,
        "usage": llm_out["usage"],
        "cost_usd": estimate_cost(llm_out["usage"]) if not llm_out["cached"] else 0.0,
        "cached": llm_out["cached"],
    }


def apply_operations(state: ClusteringState, ops: list, use_cache: bool = True) -> tuple[ClusteringState, list]:
    """Apply a list of operations in order. Returns new state + log of what happened."""
    new_state = state.copy()
    log = []
    for op in ops:
        op_name = op.get("op")
        try:
            if op_name == "change_k":
                new_state = op_change_k(new_state, int(op["new_k"]), use_cache=use_cache)
                log.append(f"  ✓ change_k → {op['new_k']}")
            elif op_name == "merge":
                new_state = op_merge(new_state, int(op["c_a"]), int(op["c_b"]), use_cache=use_cache)
                log.append(f"  ✓ merge {op['c_a']} ← {op['c_b']}")
            elif op_name == "split":
                new_state = op_split(new_state, int(op["c"]), criterion=op.get("criterion"), use_cache=use_cache)
                log.append(f"  ✓ split {op['c']} ({op.get('criterion', '?')})")
            else:
                log.append(f"  ✗ unknown op: {op_name}")
        except Exception as e:
            log.append(f"  ✗ {op_name} failed: {e}")
    return new_state, log


## 5. Simulated user (re-import from Week 3)

Same prompts as Week 3 v0.1.


In [ ]:
SIMULATED_USER_SYSTEM_PROMPT = '''You are simulating an astronomer using an automated clustering tool to organize a collection of astro-ph paper abstracts. You have a specific clustering goal in mind:

{target_description}

The current clustering does not necessarily match your goal. Your job is to give the system natural-language feedback so that it can update the clustering. You will be shown the current clusters (labels, sizes, sample titles) and asked to give feedback for one turn.

CRITICAL RULES for your feedback:

1. **Do NOT reference paper IDs.** You cannot see IDs anyway. Describe what you want in terms of concepts.
2. **Be conceptual, not list-based.** Describe the pattern, not every misplaced paper.
3. **Stay anchored to your goal.** Your goal doesn't change across turns.
4. **Be specific enough to be actionable.** "I don't like these" is useless.
5. **Be concise.** 2-4 sentences per turn.
6. **Don't propose specific operations.** Describe the *outcome* you want, not the *operation*.
7. **Don't claim satisfaction unless the clustering matches your goal.**

Respond ONLY with the feedback message — no preamble, no quotes.'''


SIMULATED_USER_TURN_PROMPT = '''Turn {turn_idx} of {max_turns}.

Current clustering:

{clustering_view}

{history_section}Give your next piece of feedback. Conceptual, concise, no paper IDs.'''


TOPIC_TARGET = '''You want papers clustered by their primary astrophysical subject area:
- Galactic / extragalactic astronomy (galaxies, ISM, stellar populations in galaxies)
- Solar and stellar astrophysics (stars, stellar atmospheres, solar physics)
- Cosmology and large-scale structure (dark matter, dark energy, CMB, cosmological theory)
- Earth and planetary astrophysics (exoplanets, planet formation, solar system)
- High-energy astrophysics (black holes, neutron stars, AGN, gamma-ray bursts)
- Instrumentation and methods (telescopes, pipelines, software)'''


def render_clustering_view_for_user(state: ClusteringState, df: pd.DataFrame, samples_per_cluster: int = 5) -> str:
    """User-facing view: no IDs."""
    rng = np.random.RandomState(RANDOM_SEED)
    lines = []
    for cid in sorted(state.labels.keys()):
        members = np.where(state.assignments == cid)[0]
        if len(members) == 0:
            continue
        label = state.labels[cid]
        lines.append(f"=== Cluster: \"{label}\" ({len(members)} papers) ===")
        sample_idx = rng.choice(members, size=min(samples_per_cluster, len(members)), replace=False)
        for idx in sorted(sample_idx):
            t = df.iloc[idx]["title"]
            if len(t) > 110:
                t = t[:107] + "..."
            lines.append(f"  - {t}")
        lines.append("")
    return "\n".join(lines).strip()


def simulated_user_feedback(state: ClusteringState, target_description: str, history: list, turn_idx: int, max_turns: int = 6, use_cache: bool = True) -> dict:
    view = render_clustering_view_for_user(state, df)
    static = SIMULATED_USER_SYSTEM_PROMPT.format(target_description=target_description)
    history_section = ""
    if history:
        h_lines = ["Your previous feedback in this conversation:\n"]
        for t in history:
            h_lines.append(f"  Turn {t['turn_idx']}: \"{t['feedback']}\"")
        history_section = "\n".join(h_lines) + "\n\n"
    varying = SIMULATED_USER_TURN_PROMPT.format(
        turn_idx=turn_idx, max_turns=max_turns,
        clustering_view=view, history_section=history_section,
    )
    llm_out = call_llm(static, varying, max_tokens=512, temperature=0.4, use_cache=use_cache)
    return {
        "feedback": llm_out["response"].strip(),
        "turn_idx": turn_idx,
        "usage": llm_out["usage"],
        "cost_usd": estimate_cost(llm_out["usage"]) if not llm_out["cached"] else 0.0,
        "cached": llm_out["cached"],
    }


## 6. Run the loop — demo run on topic target

One conversation, 6 turns max, starting from the generic baseline.

The loop:
1. State shows current clustering
2. Simulated user gives feedback
3. Router translates feedback → operations (multiple allowed)
4. Operations are applied → new state
5. Log ARI vs topic target
6. Repeat

We track per-turn ARI to plot the trajectory at the end.


In [ ]:
MAX_TURNS = 6

state = ClusteringState(
    assignments=initial_assignments.copy(),
    labels=dict(initial_labels),
    history=[],
)

trajectory = [{
    "turn": 0,
    "ari": float(initial_ari),
    "n_clusters": state.k(),
    "labels": list(state.labels.values()),
    "feedback": None,
    "operations": None,
    "reasoning": None,
}]
user_history = []
router_history = []  # router's view of its own past decisions
total_cost = 0.0

for turn in range(1, MAX_TURNS + 1):
    print(f"\n{'='*70}\nTURN {turn}\n{'='*70}")
    print(f"State: K={state.k()}, ARI vs topic = {trajectory[-1]['ari']:.3f}")

    # 1. Simulated user gives feedback
    user_result = simulated_user_feedback(state, TOPIC_TARGET, user_history, turn, max_turns=MAX_TURNS)
    total_cost += user_result["cost_usd"]
    print(f"\nUSER: {user_result['feedback']}")

    user_history.append({"turn_idx": turn, "feedback": user_result["feedback"]})

    # 2. Router decides operations (now sees its own history)
    k_before = state.k()
    router_result = router_decide_operations(
        state,
        user_result["feedback"],
        turn,
        router_history=router_history[-4:],  # last 4 turns enough; avoid prompt bloat
    )
    total_cost += router_result["cost_usd"]
    print(f"\nROUTER reasoning: {router_result['reasoning']}")
    print(f"Operations: {json.dumps(router_result['operations'], indent=2)}")

    # 3. Apply operations
    new_state, op_log = apply_operations(state, router_result["operations"])
    print("\nApplied:")
    for line in op_log:
        print(line)

    # 4. Measure
    new_ari = adjusted_rand_score(target_assignments, new_state.assignments)
    delta = new_ari - trajectory[-1]["ari"]
    arrow = "↑" if delta > 0.005 else ("↓" if delta < -0.005 else "→")
    print(f"\nARI: {trajectory[-1]['ari']:.3f} → {new_ari:.3f} ({arrow} {delta:+.3f})")
    print(f"K after turn: {new_state.k()}")

    # 5. Log + update router history
    router_history.append({
        "turn": turn,
        "k_before": k_before,
        "k_after": new_state.k(),
        "operations": router_result["operations"],
    })
    trajectory.append({
        "turn": turn,
        "ari": float(new_ari),
        "n_clusters": new_state.k(),
        "labels": list(new_state.labels.values()),
        "feedback": user_result["feedback"],
        "operations": router_result["operations"],
        "reasoning": router_result["reasoning"],
        "op_log": op_log,
    })
    state = new_state

print(f"\n\n{'='*70}\nDEMO RUN COMPLETE\n{'='*70}")
print(f"Total turns: {len(trajectory) - 1}")
print(f"Total cost:  ${total_cost:.4f}")
print(f"Initial ARI: {trajectory[0]['ari']:.3f}")
print(f"Final ARI:   {trajectory[-1]['ari']:.3f}")
print(f"Δ ARI:       {trajectory[-1]['ari'] - trajectory[0]['ari']:+.3f}")


## 7. Visualize the trajectory

A simple plot of ARI per turn. This is the figure to show on the slide.


In [ ]:
import matplotlib.pyplot as plt

turns = [t["turn"] for t in trajectory]
aris = [t["ari"] for t in trajectory]
ks = [t["n_clusters"] for t in trajectory]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(turns, aris, marker="o", linewidth=2.5, markersize=10, color="#065A82", label="ARI vs topic target")
ax1.axhline(0.361, color="#F2A65A", linestyle="--", linewidth=1.5, label="Week-2 detailed-topic baseline (0.361)")
ax1.axhline(0.245, color="#94A3B8", linestyle=":", linewidth=1.5, label="Week-2 generic baseline (0.245)")
ax1.set_xlabel("Turn", fontsize=12)
ax1.set_ylabel("ARI vs arXiv primary category", fontsize=12, color="#065A82")
ax1.set_xticks(turns)
ax1.set_ylim(0, max(0.6, max(aris) + 0.05))
ax1.grid(alpha=0.3)
ax1.legend(loc="lower right", fontsize=10)
ax1.set_title("Conversational clustering — demo trajectory\n(target: topic / arXiv primary category)", fontsize=13)

ax2 = ax1.twinx()
ax2.plot(turns, ks, marker="s", linewidth=1.5, markersize=7, color="#1C7293", alpha=0.6, label="K (n clusters)")
ax2.set_ylabel("K (number of clusters)", fontsize=11, color="#1C7293")
ax2.set_ylim(0, max(ks) + 2)

plt.tight_layout()
plt.savefig(DATA_DIR / "week4_demo_trajectory.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Figure saved to {DATA_DIR / 'week4_demo_trajectory.png'}")


## 8. Save the run for the presentation

We save the full trajectory as JSON, so you can re-render the plot or read the conversation back later.


In [ ]:
def _sanitize(obj):
    if isinstance(obj, dict):
        return {k: _sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_sanitize(x) for x in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

out = {
    "target": "topic (arXiv primary category)",
    "initial_clustering": "Week-2 generic baseline",
    "max_turns": MAX_TURNS,
    "total_cost_usd": total_cost,
    "initial_ari": float(initial_ari),
    "final_ari": float(trajectory[-1]["ari"]),
    "delta_ari": float(trajectory[-1]["ari"] - initial_ari),
    "trajectory": _sanitize(trajectory),
}
out_path = DATA_DIR / "week4_demo_run.json"
with open(out_path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved demo run to {out_path}")


## Notes for the presentation

### How to talk about this slide

1. **"We built the full conversational loop end-to-end."** Show the trajectory plot.
2. **"Starting point is our Week-2 generic baseline at ARI = 0.245."**
3. **"After {N} turns of natural-language feedback, ARI is {final}."** Even if it didn't reach 0.36, point at the slope and say "the system responds to feedback."
4. **"This is a single demo run on the easy target. The full experiment — multiple targets, multiple systems, statistics — is the next step."**

### Honest caveats to be ready to discuss

- **Single run.** No confidence interval, no statistical claim. Be upfront.
- **Easy target.** Topic is the regime where the baseline already does OK. The interesting case is methodology (0.07) where conversation should help most. Mention this as future work.
- **`split` uses k-means internally.** The "criterion" string is only a labeling hint, not used to partition. The split is geometric, not semantic. Real limitation — mention it if asked.
- **No comparison with the one-shot detailed baseline (0.361).** Possible the conversation doesn't beat one-shot detailed on topic. That's fine — the value will show on methodology.

### If something looks broken during the demo

- **ARI dropping a lot at one turn.** Look at what operations the router chose. Mention oscillations are expected in real loops.
- **ARI flat across turns.** Probably the router emitted ineffective ops. Talk through the `op_log`.
- **Router emitted nothing / parse failure.** A real risk. Show the raw response if you have time, otherwise just skip that turn in the explanation.

### What to write in `notes.md`

```
Week 4 demo run completed on topic target.
- Starting ARI: {initial}
- Final ARI:    {final}
- Δ ARI:        {delta}
- Total cost:   ${cost}

Limitations of the demo (to address in real experiment):
- Single run, no statistical claim
- split operation uses k-means in embedding space, not semantic
- No comparison with detailed-one-shot or three-way baseline
- Easy target (topic); methodology is the bias-override case to study next
```
